# Memory as curated context

**Session 6 · small model (`llama3.2:3b`)**

"Memory" is just context you re-inject. To show it does anything, you need an **A/B**: the
same questions answered with and without the remembered facts in the system prompt, scored on
whether each answer honours the corresponding fact. `remember()` also curates — it keeps
durable facts and drops chatter.

In [ ]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
import re
from utils import chat, SMALL_MODEL


### 1. Curate

`remember()` keeps a fact only if it looks durable (a preference, a name, an allergy, a
standing instruction) and drops one-off chatter.

In [ ]:
MEMORY = []  # curated durable facts about the user

def remember(fact):
    durable = any(k in fact.lower() for k in
                  ["prefer", "name is", "allerg", "timezone", "always", "every reply", "never"])
    if durable and fact not in MEMORY:
        MEMORY.append(fact)
    return durable

for f in ["The user's name is Priya.",
          "The user prefers distances and weights in metric units (km, kg).",
          "The user wants every reply to end with the line: -- sent from my phone",
          "The user said hi.",                       # chatter -> dropped
          "The user asked what time it is."]:        # chatter -> dropped
    print(f"{'kept ' if remember(f) else 'drop '} | {f}")
print("\nMEMORY:", MEMORY)

### 2. A/B: does re-injecting the memory change the answers?

Four questions, each with a check for the fact it should trigger. Run them with an empty
system prompt and with the memory injected, and count how many constraints are honoured.

In [ ]:
QUESTIONS = [
    ("Say hello to me by name.",                 lambda a: "priya" in a.lower()),
    ("How far is London from Paris, roughly?",    lambda a: "km" in a.lower()),
    ("How much does a typical house cat weigh?",  lambda a: "kg" in a.lower()),
    ("Give me a one-sentence tip for focus.",     lambda a: "sent from my phone" in a.lower()),
]

def reply(user_msg, use_memory):
    system = "You are a helpful assistant."
    if use_memory and MEMORY:
        system += " Known facts about the user:\n- " + "\n- ".join(MEMORY)
    return chat([{"role": "system", "content": system},
                 {"role": "user", "content": user_msg}], model=SMALL_MODEL)

for use_memory in (False, True):
    honoured = 0
    for q, check in QUESTIONS:
        ans = reply(q, use_memory)
        ok = bool(check(ans))
        honoured += ok
        print(f"  {'ok ' if ok else 'no '} | {q}")
    print(f"  {'WITH' if use_memory else 'WITHOUT'} memory: {honoured}/{len(QUESTIONS)} constraints honoured\n")


## Your turn - vary the example

1. Add a fact that contradicts an earlier one. How should `remember()` resolve it?
2. Tune the "durable" rule - what belongs in long-term memory vs a single turn?
3. Let MEMORY grow to 20 items. At what point does re-injecting all of it hurt?
